# 06 · Reproducción del champion (notebook limpio) y exportación de corridas

Este notebook no reutiliza nada de los anteriores: parte de cero, carga cada modelo desde el
Model Registry con `models:/ecu911-t{h}-modelo@champion`, reconstruye el hold-out con `src/` y
verifica que la PR-AUC de prueba coincide **exactamente** con la registrada en su corrida.
Al final exporta todas las corridas del experimento a `docs/corridas.csv`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1"); os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "true"); os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")

import pandas as pd
import mlflow
from mlflow import MlflowClient
from sklearn.metrics import average_precision_score

from src.datos import cargar_obt, construir_tabla_modelado, metadatos_dataset, HORIZONTES, COLUMNAS_X, RAIZ, RUTA_SNAPSHOT
from src.validacion import separar_holdout, tabla_p90, etiquetar
from src.experimento import configurar_mlflow, nombre_experimento

configurar_mlflow()
cliente = MlflowClient()
obt, meta = cargar_obt()

In [2]:
filas = []
for h in HORIZONTES:
    nombre = f"{'dev-' if os.getenv('ECU911_SINTETICO') == '1' else ''}ecu911-t{h}-modelo"
    version = cliente.get_model_version_by_alias(nombre, "champion")
    run = cliente.get_run(version.run_id)
    # el snapshot debe ser el mismo con el que se registró la corrida
    assert run.data.params["dataset_sha256"] == meta["sha256"], "El snapshot de datos no es el registrado"
    modelo = mlflow.sklearn.load_model(f"models:/{nombre}@champion")
    tabla = construir_tabla_modelado(obt, h)
    train, test = separar_holdout(tabla, h)
    y_test = etiquetar(test, tabla_p90(train))
    pr_auc = average_precision_score(y_test, modelo.predict_proba(test[COLUMNAS_X])[:, 1])
    registrada = run.data.metrics["pr_auc_test"]
    filas.append({"modelo_registrado": nombre, "version": version.version, "run_id": version.run_id,
                  "familia": run.data.tags["familia"], "pr_auc_registrada": registrada,
                  "pr_auc_reproducida": pr_auc, "diferencia": abs(pr_auc - registrada)})
rep = pd.DataFrame(filas)
assert (rep.diferencia < 1e-12).all(), "La métrica no se reproduce"
rep

,modelo_registrado,version,run_id,familia,pr_auc_registrada,pr_auc_reproducida,diferencia
0,ecu911-t3-modelo,1,8ef2ae5e31c94cdd856e635b91acf08f,boosting,0.268226,0.268226,0.0
1,ecu911-t7-modelo,1,08a437000d7a4e0ba1d84d8f16d784ab,boosting,0.255634,0.255634,0.0


## Exportación de todas las corridas (`docs/corridas.csv`)

In [3]:
corridas = mlflow.search_runs(experiment_names=[nombre_experimento()], max_results=50_000)
cols = (["run_id", "tags.mlflow.runName", "tags.fase", "tags.familia", "tags.horizonte", "tags.git_commit",
         "tags.mlflow.parentRunId", "status", "start_time", "end_time"]
        + sorted(c for c in corridas.columns if c.startswith("metrics."))
        + sorted(c for c in corridas.columns if c.startswith("params.")))
corridas = corridas[[c for c in cols if c in corridas.columns]].sort_values("start_time")
destino = RAIZ / ("data/dev" if os.getenv("ECU911_SINTETICO") == "1" else "docs") / "corridas.csv"
corridas.to_csv(destino, index=False)
print(f"{len(corridas)} corridas → {destino.relative_to(RAIZ)}")
corridas.groupby(["tags.horizonte", "tags.fase"]).size().unstack(fill_value=0)

206 corridas → docs/corridas.csv


tags.fase,baseline,candidato,final,llm,tuning
tags.horizonte,,,,,
t3,2,5,2,1,93
t7,2,5,2,1,93
